In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertModel, AdamW, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
import pickle as pkl
from tqdm.notebook import tqdm
import numpy as np
import scipy as sp
import shap
import re

In [18]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
os.environ['TOKENIZERS_PARALLELISM'] = 'false' # there might be interferences with the parallelism of the Hugging Face Trainer
os.environ['WANDB_DISABLED'] = "true"

print(f"Using device: {device}")
if torch.device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

Using device: cpu


In [3]:
filename='data/k_fold_data.xlsx'
df = pd.read_excel(filename, index_col=0)
df.shape

(88219, 9)

In [17]:
kf_df= df[df['k_fold']==5]

In [5]:
speech_party_map= dict(zip(df['speechnumber'],df['party']))

In [6]:
"""takes a text and returns a list of texts in given length"""
def chunksspeech(text,term, length):
    return [' '.join(chunk) for chunk in list((text[0+i:length+i] for i in range(0, int(term), length)))]

In [7]:
def chunk_data(texts,labels,speech_ids,length):
    chunked_speeches=[chunksspeech(text.split(" "),len(text.split(" ")),length) for text in texts]
    speech_chunks = [chunk for speech in chunked_speeches for chunk in speech]
    chunk_labels = [label for label,speech in zip(labels,chunked_speeches) for _ in speech]
    chunk_to_speech_mapping = [speech_id for speech_id,speech in zip(speech_ids,chunked_speeches) for _ in speech]
    return speech_chunks,chunk_labels,chunk_to_speech_mapping

In [8]:
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, ids, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.ids = ids
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        speech_id = self.ids[idx]
        encoding = self.tokenizer(text, return_tensors='pt', max_length=self.max_length, padding='max_length', truncation=True)
        return {'input_ids': encoding['input_ids'].flatten(), 'attention_mask': encoding['attention_mask'].flatten(), 'label': torch.tensor(label,dtype=torch.long), 'speech_id': speech_id}

In [9]:
'''
BERT Classifier with a BERT layer, Dropout layer and a linear layer
'''
class BERTClassifier(nn.Module):
    def __init__(self, bert_model_name, num_classes):
        super(BERTClassifier, self).__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        x = self.dropout(pooled_output)
        logits = self.fc(x)
        return logits

In [10]:
def train(model, data_loader, optimizer, scheduler, device):
    model.train()
    cross_entropy_loss = 0
    for batch in tqdm(data_loader,desc="Training"):
        # Reset gradients before first run - if training
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        # forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = nn.CrossEntropyLoss()(outputs, labels)
        # backpropagation with optimizer step
        loss.backward()
        cross_entropy_loss += loss.item()
        optimizer.step()
        scheduler.step()
    cross_entropy_loss = cross_entropy_loss/len(data_loader)
    return cross_entropy_loss

In [11]:
def evaluate(model, data_loader, device, ids=None):
    model.eval()
    predictions = []
    actual_labels = []
    all_probs = []
    all_ids = []  # collect speech Id to aggregate later
    with torch.no_grad():
        for batch in tqdm(data_loader,desc="Evaluation"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            _, preds = torch.max(outputs, dim=1)
            probs = nn.functional.softmax(outputs, dim=1)
            predictions.extend(preds.cpu().tolist())
            actual_labels.extend(labels.cpu().tolist())
            all_probs.extend(probs.cpu().tolist())
            all_ids.extend(batch['speech_id'])  # edit to process chunks#'speech_id'
    return accuracy_score(actual_labels, predictions),classification_report(actual_labels, predictions), all_probs, all_ids #classification_report(actual_labels, predictions)

In [12]:
"""Creates a dataframe with speech_id and predicitions on Chunk-level and aggregates to speech-level """

def speech_chunk_dataframe(prob, speech_ids):
    prob_df = pd.DataFrame(prob)
    speech_ids = [x.item() for x in speech_ids]
    prob_df['speechnumber'] = speech_ids
    return prob_df

"Group chunk-level probabilites by speech_id and compute weighted mean if provided"

def group_probablities_by_speech(probabilities_df, weights=None):
  grouped_probabilities = probabilities_df.groupby('speechnumber').mean()
  return grouped_probabilities

'''Evaluate on speech level'''

def evaluate_speeches(probabilities_df, speech_id_to_party_map, weights=None):
    grouped_probabilities = group_probablities_by_speech(probabilities_df, weights)
    # get highest average probability per speech
    predicted_speech_labels = grouped_probabilities.idxmax(axis=1).tolist()
    # get true labels
    grouped_speech_ids = grouped_probabilities.index.tolist()
    true_labels= [speech_id_to_party_map[k] for k in grouped_speech_ids]
    # get accuracy and report
    accuracy = accuracy_score(true_labels, predicted_speech_labels)
    report = classification_report(true_labels, predicted_speech_labels)

    return  accuracy, report,true_labels, predicted_speech_labels,grouped_probabilities

In [13]:
'''sentence splitter'''

def sentence_splitter(text):
    sentences= re.split(r'(?<=[.!?])\s+', text.strip())
    return [s for s in sentences if s]

In [14]:
def predict(texts):
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs, dim=1)
    return probs.detach().cpu().numpy()

In [15]:
# Set up parameters
bert_model_name= 'bert-base-german-cased'
num_classes = 9
max_length = 256
batch_size = 20
num_epochs = 2
learning_rate = 2e-5
tokenizer = BertTokenizer.from_pretrained(bert_model_name)

C:\Users\sarah\anaconda3\lib\site-packages\huggingface_hub\file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [21]:
PATH = "data/bert_classifier_k_fold_5.pth"

model = BERTClassifier(bert_model_name, num_classes).to(device)
state_dict =torch.load(PATH, weights_only=False, map_location=torch.device('cpu'))
model.load_state_dict(state_dict, strict=False)

Some weights of the model checkpoint at bert-base-german-cased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


_IncompatibleKeys(missing_keys=['bert.embeddings.position_ids'], unexpected_keys=[])

In [ ]:
df_val = kf_df
speech_party_map= dict(zip(df_val['speechnumber'],df_val['party']))

v_texts=list(df_val['text'])
v_labels=list(df_val['party'])
v_mps=list(df_val['speaker'])
v_speech_ids=list(df_val['speechnumber'])

val_texts, val_labels, val_ids = chunk_data(v_texts,v_labels,v_speech_ids,max_length)
val_dataset = TextClassificationDataset(val_texts, val_labels, val_ids, tokenizer, max_length)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size)

accuracy, report, probabilities, speech_ids = evaluate(model, val_dataloader, device)

In [ ]:
# Aggregate on speech-level
speech_level_df = speech_chunk_dataframe(probabilities, speech_ids)
speech_accuracy, speech_report, true_speech_labels, predicted_speech_labels, speech_probabilities = evaluate_speeches(speech_level_df, speech_party_map, weights=None)

In [ ]:
# https://shap.readthedocs.io/en/latest/example_notebooks/text_examples/sentiment_analysis/Emotion%20classification%20multiclass%20example.html

# build a pipeline object to do predictions
pred = transformers.pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0,
    return_all_scores=True,
)

explainer = shap.Explainer(pred)

shap_values = explainer(data["text"][:3])

shap.plots.text(shap_values)

In [ ]:
def predict(texts):
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs, dim=1)
    return probs.detach().cpu().numpy()


In [ ]:
masker = shap.maskers.Text(tokenizer=split_into_sentences)

In [ ]:
explainer = shap.Explainer(predict, masker, algorithm="partition")

In [ ]:
text = "This movie was amazing. The acting was superb. But the ending was disappointing."
shap_values = explainer([text])
shap.plots.text(shap_values[0])